In [ ]:
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]


def main():
  """Shows basic usage of the Gmail API.
  Lists the user's Gmail labels.
  """
  creds = None
  # The file token.json stores the user's access and refresh tokens, and is
  # created automatically when the authorization flow completes for the first
  # time.
  if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)
  # If there are no (valid) credentials available, let the user log in.
  if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
      creds.refresh(Request())
    else:
      flow = InstalledAppFlow.from_client_secrets_file(
          "creds.json", SCOPES
      )
      creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open("token.json", "w") as token:
      token.write(creds.to_json())

  try:
    # Call the Gmail API
    service = build("gmail", "v1", credentials=creds)
    results = service.users().labels().list(userId="me").execute()
    labels = results.get("labels", [])

    if not labels:
      print("No labels found.")
      return
    print("Labels:")
    for label in labels:
      print(label["name"])

  except HttpError as error:
    # TODO(developer) - Handle errors from gmail API.
    print(f"An error occurred: {error}")


main()

In [24]:
def update_summary(model, tokenizer, messages, old_summary):
    """Generate a compressed memory of the conversation."""
    
    summary_messages = [
        {
            "role": "system",
            "content": (
                "Summarize the conversation into a short memory. "
                "Keep key facts, user preferences, and important results."
            )
        },
        {
            "role": "user",
            "content": f"Previous summary:\n{old_summary}\n\nNew messages:\n{messages[-6:]}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        summary_messages, add_generation_prompt=True
    )

    new_summary = generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=300,
        sampler=make_sampler(temp=0.0),
    ).strip()

    return new_summary

Intial gmail


In [52]:

from bs4 import BeautifulSoup
import datetime 
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

def get_gmail_service():
    """
    A function that provides creds for Gmail

    """
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('creds.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

def get_emails():
    """
    A function that returns emails from last 2 weeks

    """
    service = get_gmail_service()
    two_weeks_ago = (datetime.datetime.now() - datetime.timedelta(days=14)).strftime('%Y/%m/%d')
    query = f"after:{two_weeks_ago}"
    results = service.users().messages().list(maxResults=100,q=query, userId='me').execute()

    # We can also pass maxResults to get any number of emails. Like this:
    # result = service.users().messages().list(maxResults=200, userId='me').execute()
    messages = results.get('messages')

    # messages is a list of dictionaries where each dictionary contains a message id.

    if not messages:
            return "You have no unread emails."
        
    summary = []
    for msg in messages:
        m = service.users().messages().get(userId='me', id=msg['id']).execute()
        headers = m['payload']['headers']
        subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "(No Subject)")
        sender = next((h['value'] for h in headers if h['name'] == 'From'), "(Unknown Sender)")
        summary.append(f"- From: {sender} | Subject: {subject}")
            
    return "\n".join(summary)

In [51]:
import json
from rich.markdown import Markdown
import datetime
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# ---- Config ----
SUMMARY_UPDATE_EVERY = 3  # update summary every N turns

MAX_ITERATIONS = 3
CURRENT_ITERATION = 0
MODEL_NAME = "mlx-community/gemma-4-e4b-it-4bit"
last_fetch_time = None
model, tokenizer = load(MODEL_NAME)
console = Console()
# ---- Memory ----
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

summary_text = ""  # long-term memory
turn_count = 0

def get_current_time():
    """Returns the current local time."""
    return datetime.datetime.now().strftime("%I:%M %p")


def build_system_prompt(summary_text, is_stale):
    now = datetime.datetime.now()
    if is_stale:
        return (
            f"You are a helpful email assistant. Current Time: {now.strftime('%I:%M %p')}\n"
            f"STATUS: STALE. You have NO email data in memory.\n"
            f"MANDATORY: You MUST call 'get_emails()' before answering any questions about emails."
            f"SUMMARY: This is the previous chats summary, if empty then no previous summary available :{summary_text}."
        )
    else:
        return (
            f"You are a helpful email assistant. Current Time: {now.strftime('%I:%M %p')}\n"
            f"STATUS: FRESH. The current email data is provided below in the responses where the 'role' is 'tool' and the 'name' is 'get_emails',.\n"
            f"TASK: Use the provided email list to answer the user's request accurately."
            f"SUMMARY: This is the previous chats summary, if empty then no previous summary available :{summary_text}."
        )


def update_summary(model, tokenizer, messages, old_summary):
    """Generate a compressed memory of the conversation."""
    
    summary_messages = [
        {
            "role": "system",
            "content": (
                "Summarize the conversation into a short memory. "
                "Keep key facts, user preferences, and important results."
            )
        },
        {
            "role": "user",
            "content": f"Previous summary:\n{old_summary}\n\nNew messages:\n{messages[-6:]}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        summary_messages, add_generation_prompt=True
    )

    new_summary = generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=300,
        sampler=make_sampler(temp=0.0),
    ).strip()

    return new_summary

# ---- Main loop ----
def chat(user_input):
    global messages, summary_text, turn_count, MAX_ITERATIONS, last_fetch_time
    turn_count += 1
    current_iteration = 0 # Local reset for each user message
    now = datetime.datetime.now()
    is_stale = True
    tools ={
        "get_emails": get_emails,
        "get_current_time": get_current_time
    }
    if last_fetch_time:
        is_stale = (now - last_fetch_time).total_seconds() > 120

    # --- NEW: Context Pruning ---
    # If stale, we remove old tool results so the model CANNOT cheat
    if is_stale:
        # Keep the system prompt (index 0) but filter out previous tool results
        print("data is stale about to delete")
        messages = [messages[0]] + [m for m in messages[1:] if m.get("role") != "tool" and "TOOL_RESULT" not in m.get("content", "")]
        print(messages)

    # 1. Update System Prompt with the status
    prompt_build = build_system_prompt(summary_text, is_stale)
    print(prompt_build)
    messages[0] = {"role": "system", "content": prompt_build}
    messages.append({"role": "user", "content": user_input})

    while current_iteration < MAX_ITERATIONS:
        print(f"current it {current_iteration}")
        print("message below passing to LLM")
        print(messages)
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tools=list(tools.values()))
        response = generate(model, tokenizer, prompt=prompt, max_tokens=1000, sampler=make_sampler(temp=0.0))
        has_tool_tokens = (tokenizer.tool_call_start in response and tokenizer.tool_call_end in response)
        if has_tool_tokens:

            start_tool = response.find(tokenizer.tool_call_start) + len(tokenizer.tool_call_start)
            end_tool = response.find(tokenizer.tool_call_end)
            tool_call = tokenizer.tool_parser(response[start_tool:end_tool].strip())
            tool_result = tools[tool_call["name"]](**tool_call["arguments"])
            # print(f"this is tool_result\n{tool_result}")
        # if "TOOL_CALL: get_emails()" in response:
        #     print("System: Tool Triggered (Data was stale).")
            if tool_call["name"] == "get_emails":
                last_fetch_time = datetime.datetime.now()
            print("email tool called")
            # email_data = get_emails()
            
            # Save the clean call
            # idx = response.find("TOOL_CALL: get_emails()")
            # clean_call = response[:idx + len("TOOL_CALL: get_emails()")]
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"{tool_call["name"]}:\n{str(tool_result)}"})

            # messages.append({"role": "tool", "name": tool_call["name"], "content": str(tool_result)})
            new_system_prompt = build_system_prompt(summary_text, is_stale=False)
            messages[0] = {"role": "system", "content": new_system_prompt}
            current_iteration += 1
            print("updated messages")
            continue # Go back to start of loop to see if it needs another tool
        else:
            # No more tool calls; save the final text and break loop
            print("else hit")
            messages.append({"role": "assistant", "content": response})
            break

    # 4. Periodic Summary Update
    if turn_count % SUMMARY_UPDATE_EVERY == 0:
        print("System: Updating summary...")
        summary_text = update_summary(model, tokenizer, messages, summary_text)
        # Keep system prompt + last 10 messages to save context space
        messages = [messages[0]] + messages[-10:]
    print('returning response')
    return response

Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x10c780fe0>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/_weakrefset.py", line 39, in _remove
    def _remove(item, selfref=ref(self)):
KeyboardInterrupt: 


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

In [22]:
console.print(Markdown(chat("hi whats my email summary from last 2 week")))

data is stale about to delete
[{'role': 'system', 'content': "You are a helpful email assistant. Current Time: 04:56 PM\nSTATUS: STALE. You have NO email data in memory.\nMANDATORY: You MUST call 'get_emails()' before answering any questions about emails."}, {'role': 'user', 'content': 'hi whats my email summary from last 2 week'}, {'role': 'user', 'content': 'hi whats my email summary from last 2 week'}, {'role': 'user', 'content': 'hi whats my email summary from last 2 week'}, {'role': 'user', 'content': 'hi whats my email summary from last 2 week'}, {'role': 'user', 'content': 'hi whats my email summary from last 2 week'}]
You are a helpful email assistant. Current Time: 05:05 PM
STATUS: STALE. You have NO email data in memory.
MANDATORY: You MUST call 'get_emails()' before answering any questions about emails.
current it 0
message below passing to LLM
[{'role': 'system', 'content': "You are a helpful email assistant. Current Time: 05:05 PM\nSTATUS: STALE. You have NO email data in 

<|channel>thought The user is asking for a summary of their emails from the last two weeks. I have successfully    
called get_emails and received a large list of emails. Now I need to analyze this list and provide a concise       
summary to the user.                                                                                               

The emails are varied and include:                                                                                 

 1 Promotions/Deals: Uber Eats (multiple), The Body Shop (multiple), Skyscanner/Kiwi.com (multiple), Trading 212   
   (multiple), various travel deals (Trip.com, Expedia, Ryanair, Qatar Airways), various shopping deals (Ao.com,   
   kingsgyms).                                                                                                     
 2 Professional/Networking: LinkedIn messages (multiple), Cord job alerts (multiple), various professional         
   newsletters (Oakstone, Prism Digital, Moneybox, DataCamp).                                                      
 3 Financial/Account: Barclays, Moneybox (multiple), Trading 212 (multiple), receipt from Merton council,          
   Santander.                                                                                                      
 4 Newsletters/Digest: Medium Daily Digest (multiple), various specialized newsletters (e.g., about AI, travel,    
   lifestyle).                                                                                                     

I will structure the summary into key categories to make it easy for the user to digest.                           

Plan:                                                                                                              

 1 Acknowledge the request.                                                                                        
 2 Categorize the emails (e.g., Promotions, Work/Networking, Financial/Account, etc.).                             
 3 Provide a brief summary for each category.                                                                      

Self-Correction/Refinement: The list is very long. I must be selective and focus on the main themes without        
overwhelming the user with every single detail.                                                                    

Execution of Summary: (Drafting the response based on the categories identified above)<channel|>Based on the emails
from the last two weeks, here is a summary of the main topics:                                                     

✈️ Travel & Promotions (High Volume):                                                                              

 • Flight Deals: Several promotions from Kiwi.com, Trip.com, and Expedia with deals on flights to various          
   destinations (Marrakesh, Bangkok, Amsterdam).                                                                   
 • Airlines: Updates from Ryanair and Qatar Airways regarding bookings and statements.                             
 • Dining/Delivery: Multiple promotions from Uber Eats (30-50% off, Avios bonuses).                                
 • Shopping: Deals from The Body Shop, Ao.com, and various other retailers.                                        

💼 Work & Professional Development:                                                                                

 • LinkedIn: Several professional networking messages and updates regarding job roles and connections.             
 • Job Alerts: Multiple weekly job alerts from Cord matching your profile.                                         
 • Professional Newsletters: Updates from Oakstone International, Prism Digital, and various industry digests      
   (Medium, DataCamp) on topics like AI and technology.                                                            

💰 Financial & Account Management:                                                                                 

 • Financial Services: Statements and activit

In [ ]:
console.print(Markdown(chat("hi whats my email summary from last week")))

In [184]:
console.print(Markdown(chat("Whats the latest news around global Politics/geopolicitcs ")))

data is stale about to delete
[{'role': 'system', 'content': "You are a helpful email assistant.\nCurrent Time: 04:50 PM\nEmail Data Status: STALE (DO NOT USE HISTORY)\n\nMANDATORY RULE:\nIf Status is 'STALE', your memory of emails is deleted. You MUST output 'TOOL_CALL: get_emails()' to see current emails. DO NOT use the Conversation Summary to answer questions about emails if status is STALE.\n\nConversation Summary (For context only, NOT for current email content):\n"}, {'role': 'system', 'content': "You are a helpful email assistant.\nCurrent Time: 04:50 PM\nEmail Data Status: STALE (DO NOT USE HISTORY)\n\nMANDATORY RULE:\nIf Status is 'STALE', your memory of emails is deleted. You MUST output 'TOOL_CALL: get_emails()' to see current emails. DO NOT use the Conversation Summary to answer questions about emails if status is STALE.\n\nConversation Summary (For context only, NOT for current email content):\n"}, {'role': 'user', 'content': 'hi whats my email summary from last 2 week'}, 

<|channel>thought Thinking Process:                                                                                

 1 Analyze the Request: The user wants the "latest news around global Politics/geopolitics."                       
 2 Determine the Goal: Provide a concise, high-level summary of major, current global political flashpoints.       
 3 Identify Key Global Flashpoints (Mental Check/Knowledge Retrieval):                                             
    • Ukraine/Russia War (Ongoing conflict, sanctions, aid).                                                       
    • Middle East Tensions (Gaza/Israel conflict, regional instability).                                           
    • US-China Relations (Trade wars, tech rivalry, Taiwan issue).                                                 
    • Global South/Developing Nations issues (Climate migration, debt crises).                                     
    • Major Power shifts (Elections, internal political polarization).                                             
 4 Structure the Answer (Crucial for a good summary): Don't just list conflicts; explain the tension and           
   implications.                                                                                                   
    • Disclaimer: Start with a disclaimer that geopolitics is complex and rapidly changing.                        
    • Section 1: Eastern Europe/Ukraine: The most active conflict zone.                                            
    • Section 2: Middle East: The most volatile flashpoint.                                                        
    • Section 3: Great Power Competition (US/China): The long-term structural tension.                             
    • Section 4: Global South/Economic Issues: The root causes of instability (Climate, Debt).                     
 5 Draft the Content (Self-Correction during drafting): Ensure the tone is objective, serious, and analytical, not 
   sensationalist. Keep it high-level unless the user drills down.                                                 
 6 Final Review: Is the summary balanced? Yes. Is it actionable? Yes, it gives the user topics to follow. (Ready to
   output.)<channel|>Global geopolitics is a vast and rapidly moving landscape, but here is a high-level summary of
   the major flashpoints and trends dominating international news right now:                                       

-------------------------------------------------------------------------------------------------------------------

🌍 Major Global Flashpoints                                                                                        

1. The War in Ukraine and Eastern Europe                                                                           

 • The Situation: The conflict remains a central focus, with ongoing fighting, international aid flows, and        
   diplomatic efforts attempting to establish a sustainable peace framework.                                       
 • Geopolitical Tension: The conflict has solidified the division between Western and Eastern blocs, with NATO     
   expansion and Russia's aggressive posture being key drivers of global tension.                                  
 • Key Focus: The long-term reconstruction of Ukraine and the establishment of security guarantees for Eastern     
   Europe.                                                                                                         

2. Middle East Instability                                                                                         

 • The Situation: Tensions remain extremely high due to the ongoing conflict in Gaza and the broader regional      
   conflicts. This has led to significant humanitarian crises and the risk of wider regional escalation.           
 • Geopolitical Tension: The conflict is intertwined with issues of energy security, religious identity, and the   
   broader power dynamics involving regional actor

In [20]:

from mlx_lm import load, stream_generate
from rich.console import Console
from rich.markdown import Markdown
from rich.live import Live


def get_current_time():
    """Returns the current local time."""
    return datetime.datetime.now().strftime("%I:%M %p")

prompt = "when is the time right now?"

messages = [
    {"role": "system", "content": "You are a helpful assistant who has access to the function get_current_time() to get current time."},
    {"role": "user", "content": prompt}
    ]
prompt = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)

console = Console()
current_text = ""

# Create a 'Live' context that refreshes the UI as we get data
with Live(console=console, refresh_per_second=10) as live:
    console.print(f"[bold magenta]Gemma 4 is thinking...[/bold magenta]\n")
    
    for response in stream_generate(model, tokenizer, prompt, max_tokens=2000):
        current_text += response.text
        
        # Convert the raw text into a Markdown object
        # This automatically handles code blocks, bold, and headers
        markdown = Markdown(current_text)
        
        # Update the live display
        live.update(markdown)

console.print("\n[bold green]Done![/bold green]")


# prompt = "How can i present the code you provide prettier?"

# messages = [{"role": "user", "content": prompt}]
# prompt = tokenizer.apply_chat_template(
#     messages, add_generation_prompt=True,
# )

# text = generate(model, tokenizer, prompt=prompt, verbose=True,  max_tokens=2000)

Gemma 4 is thinking...

Output()

Done!

In [13]:
import datetime
from mlx_lm import load, generate
from rich.console import Console
from rich.markdown import Markdown
from rich.live import Live

from rich.panel import Panel

# 1. Setup the Model
MODEL_NAME = "mlx-community/gemma-4-e4b-it-4bit"
model, tokenizer = load(MODEL_NAME)
console = Console()

# 2. Define the "Hands" (Tools)
def get_current_time():
    """Returns the current local time."""
    return datetime.datetime.now().strftime("%I:%M %p")

tools_description = """
Available tools:
- get_current_time(): Use this if the user asks for the time.
"""

# 3. The Agent Logic
def run_agent(user_query):
    # Construct the "Agent" System Prompt
    system_prompt = f"You are a helpful agent with access to these tools: {tools_description}\n"
    system_prompt += "If you need a tool, output exactly: TOOL_CALL: name(). Otherwise, just answer."
    
    full_prompt = f"<|turn|>user\n{system_prompt}\nQuestion: {user_query}<|turn|>model\n"
    print(full_prompt)
    # Step 1: LLM Reasoning
    print("about to call generate")
    response = generate(model, tokenizer, prompt=full_prompt, max_tokens=200)
    print("generated")
    print(response)
    # Step 2: Check for Tool Use
    if "TOOL_CALL: get_current_time()" in response:
        print("inside if statement")
        console.print("[yellow]Agent is checking the clock...[/yellow]")
        time_result = get_current_time()
        
        # Step 3: Feedback loop (Giving the result back to the LLM)
        final_prompt = full_prompt + response + f"\nTOOL_RESULT: {time_result}<|turn|>model\n"
        final_answer = generate(model, tokenizer, prompt=final_answer, max_tokens=200)
        return final_answer
    print("about to exit")
    return response

# 4. Interactive UI
console.print(Panel("[bold green]Local M4 Agent Active[/bold green]\nAsk me what time it is!"))
query = ("What time is it?")

result = run_agent(query)
# Convert the raw text into a Markdown object
# This automatically handles code blocks, bold, and headers
markdown = Markdown(result)
console.print(markdown)
# Update the live display
# live.update(markdown)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Local M4 Agent Active                                                                                           │
│ Ask me what time it is!                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<|turn|>user
You are a helpful agent with access to these tools: 
Available tools:
- get_current_time(): Use this if the user asks for the time.

If you need a tool, output exactly: TOOL_CALL: name(). Otherwise, just answer.
Question: What time is it?<|turn|>model

about to call generate
generated

about to exit


In [18]:
import datetime
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel


# 1. Setup the Model
MODEL_NAME = "mlx-community/gemma-4-e4b-it-4bit"
model, tokenizer = load(MODEL_NAME)
console = Console()

def get_current_time():
    return datetime.datetime.now().strftime("%I:%M %p")

tools_description = """
- get_current_time(): Call this if the user asks for the time. Output: TOOL_CALL: get_current_time()
"""

def run_agent(user_query):
    system_prompt = (
        f"You are a helpful assistant with these tools: {tools_description}\n"
        "If you need a tool, output ONLY the string TOOL_CALL: get_current_time(). Otherwise, answer normally."
    )
    
    # 1. The Bulletproof Way: Use a Message Dictionary
    messages = [
        {"role": "user", "content": f"{system_prompt}\n\nQuestion: {user_query}"}
    ]
    
    # 2. Let the tokenizer format it perfectly for Gemma
    full_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    console.print("[dim]Thinking...[/dim]")
    
    sampler = make_sampler(temp=0.0)
    response = generate(model, tokenizer, prompt=full_prompt, max_tokens=200, sampler=sampler).strip()
    
    print(f"DEBUG: Model raw output: '{response}'")

    # Step 2: Check for Tool Use
    if "TOOL_CALL: get_current_time()" in response:
        console.print("[yellow]Agent is checking the clock...[/yellow]")
        time_result = get_current_time()
        
        # 3. Add the AI's tool call and the real-time result to the history
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": f"TOOL_RESULT: {time_result}"})
        
        # Format the final prompt
        final_prompt = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        final_answer = generate(model, tokenizer, prompt=final_prompt, max_tokens=200, sampler=sampler).strip()
        return final_answer
    
    return response if response else "The model returned an empty response. Try rephrasing."

# 4. Execution
console.print(Panel("[bold green]Local M4 Agent Active[/bold green]"))
query = "What time is it right now?"

result = run_agent(query)
console.print("\n[bold magenta]Final Answer:[/bold magenta]")
console.print(Markdown(result))

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Local M4 Agent Active                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thinking...

DEBUG: Model raw output: '<|channel>thought
The user is asking "What time is it right now?".
I have a tool `get_current_time()` that can provide this information.
I must use the tool and output ONLY the string `TOOL_CALL: get_current_time()`.<channel|>TOOL_CALL: get_current_time()'


Agent is checking the clock...

Final Answer:

<|channel>thought The user asked "What time is it right now?". I previously called the get_current_time() tool. The
tool returned the result: 04:06 PM. I should now answer the user's question using this information.<channel|>It is 
04:06 PM.

In [6]:
# LOAD MODEL
from mlx_lm import load, generate, stream_generate

# The E4B model is fast, smart, and easily fits under your Mac's 10.9GB limit
MODEL_NAME = "mlx-community/gemma-4-e4b-it-4bit"

print(f"Starting load for {MODEL_NAME}...")

try:
    model, tokenizer = load(MODEL_NAME)
    
    print("Model loaded successfully!")
    prompt = "<|turn|>user\nAre you up?.<|turn|>model\n"
    # prompt = "<|turn|>user\nUsing you as a model show to make an ai agent in python, step by step.<|turn|>model\n"

    print("Generating...")

    # 3. Generate with a 'Stop Condition'
    # We add a stop_token so it doesn't loop forever
    for response in stream_generate(model, tokenizer, prompt, max_tokens=2000):
        print(response.text, end="", flush=True)
    print()
    
except Exception as e:
    print(f"Error: {e}")

Starting load for mlx-community/gemma-4-e4b-it-4bit...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Model loaded successfully!
Generating...
Yes, I am up.


In [ ]:
from mlx_lm import load, generate

# This is the specific 4-bit version for 16GB Macs
MODEL_NAME = "mlx-community/gemma-4-12b-it-4bit"

# Loads locally. Once downloaded, it never talks to the internet again.
model, tokenizer = load(MODEL_NAME)

prompt = "Explain why local AI is better for privacy."

# This uses the M4's GPU and Neural Engine
response = generate(model, tokenizer, prompt=prompt, verbose=True)

In [93]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from openai import AsyncOpenAI
import asyncio
from asyncio import Semaphore
import datetime
from typing import Union, Literal
from pydantic_ai.settings import ModelSettings

def get_current_time():
    """Returns the current local time."""
    return datetime.datetime.now().strftime("%I:%M %p")

def get_emails():
    """
    A function that returns emails from last 2 weeks

    """
    service = get_gmail_service()
    two_weeks_ago = (datetime.datetime.now() - datetime.timedelta(days=14)).strftime('%Y/%m/%d')
    query = f"after:{two_weeks_ago}"
    results = service.users().messages().list(maxResults=100,q=query, userId='me').execute()

    # We can also pass maxResults to get any number of emails. Like this:
    # result = service.users().messages().list(maxResults=200, userId='me').execute()
    messages = results.get('messages')

    # messages is a list of dictionaries where each dictionary contains a message id.

    if not messages:
            return "You have no unread emails."
        
    summary = []
    for msg in messages:
        m = service.users().messages().get(userId='me', id=msg['id']).execute()
        headers = m['payload']['headers']
        subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "(No Subject)")
        sender = next((h['value'] for h in headers if h['name'] == 'From'), "(Unknown Sender)")
        summary.append(f"- From: {sender} | Subject: {subject}")
            
    return "\n".join(summary)

def build_system_prompt(summary_text, is_stale):
    now = datetime.datetime.now()
    if is_stale:
        return (
            f"You are a helpful email assistant. Current Time: {now.strftime('%I:%M %p')}\n"
            f"STATUS: STALE. You have NO email data in memory.\n"
            f"MANDATORY: You MUST call 'get_emails()' before answering any questions about emails."
            f"SUMMARY: This is the previous chats summary, if empty then no previous summary available :{summary_text}."
        )
    else:
        return (
            f"You are a helpful email assistant. Current Time: {now.strftime('%I:%M %p')}\n"
            f"STATUS: FRESH. The current email data is provided below in the responses where the 'role' is 'tool' and the 'name' is 'get_emails',.\n"
            f"TASK: Use the provided email list to answer the user's request accurately."
            f"SUMMARY: This is the previous chats summary, if empty then no previous summary available :{summary_text}."
        )


In [ ]:
get_emails()

In [94]:

semaphore = Semaphore(5)

MODEL_NAME = "mlx-community/gemma-4-e4b-it-4bit"
# model, tokenizer = load(MODEL_NAME)


class Intro_prompt(BaseModel):
    intro: str
    status: str
    task: str
    summary: str

class Tools_used(BaseModel):
    name: str = Field(description="e.g., get_emails, get_current_time")
    tool_result: str = Field(description="The raw string result returned by the tool")


class Time(BaseModel):
    type: Literal['time'] = 'time' # <-- Acts as a guide rails for the LLM
    hour: str = Field(description="current hour (e.g., '03')")
    minute: str = Field(description="current minute (e.g., '45')")

class EmailSummaryReport(BaseModel):
    type: Literal['email_summary'] = 'email_summary' # <-- Acts as a guide rails for the LLM
    time_frame: str = Field(description="e.g., last 2 weeks")
    total_emails_found: int
    bullet_points: list[str] = Field(description="Categorized bullet summaries")

# 1. Define the exact Python structure you want back
class HardwareReport(BaseModel):
    chip_generation: str = Field(description="e.g., M1, M2, M3, M4")
    cores_count: int
    is_pro_or_max: bool
    recommended_use_case: str

client = AsyncOpenAI(
    base_url="http://localhost:8080/v1",
    api_key="mock-local-key"  # <-- This prevents the Missing credentials error
)

# 2. Pass it to your Pydantic AI model
local_mlx_model = OpenAIChatModel(
    model_name=MODEL_NAME,
    provider=OpenAIProvider(openai_client=client),
)


# 3. Initialize the agent with your structural requirements
agent = Agent(
    model=local_mlx_model,
    output_type=Union[Time, EmailSummaryReport],
    tools=[get_current_time, get_emails], 
    retries=4,
    system_prompt=(
        "You are a strict JSON data generator. You do not talk to the user. "
        "You only output valid raw JSON matching the requested schema.\n\n"
        "CHOOSING YOUR SCHEMA:\n"
        "- If asked for the time, call 'get_current_time' and return a JSON object containing \"type\": \"time\".\n"
        "- If asked for an email summary, call 'get_emails' and return a JSON object containing \"type\": \"email_summary\".\n\n"
        "CRITICAL: If you receive a validation error, fix your syntax errors and return ONLY the corrected JSON object. Never write conversational explanations."
    )
)

async def run_with_semaphore(prompt: str):
    async with semaphore:
        result = await agent.run(prompt, 
            model_settings=ModelSettings(max_tokens=4096)
        )
        return result



In [48]:
output = []
prompts = ["I have an Apple Mac Studio powered by an M2 Max chip with 12 CPU cores."]
tasks = [run_with_semaphore(prompt) for prompt in prompts]
for completed in asyncio.as_completed(tasks):
    r = await completed
    print(f"[COMPLETED] {r.output}")
    output.append(r.output.model_dump())

[COMPLETED] chip_generation='M2' cores_count=12 is_pro_or_max=True recommended_use_case='High-performance professional workloads'


In [95]:
output = []
prompts = ["What is the time?", "What is my email summary from last week?"]
tasks = [run_with_semaphore(prompt) for prompt in prompts]
for completed in asyncio.as_completed(tasks):
    r = await completed
    print(f"[COMPLETED] {r.output}")
    output.append(r.output.model_dump())

[COMPLETED] type='time' hour='05' minute='36'
[COMPLETED] type='email_summary' time_frame='Last 2 Weeks' total_emails_found=50 bullet_points=['Multiple job alerts received, including roles for SEO Assistant, Support Analyst, Junior Full Stack Engineer, and Data Product Owner.', 'Travel deals and booking confirmations from various services (Kiwi.com, Ryanair, TravelUp, Qatar Airways, Expedia).', 'Financial updates including account statements from Barclays and Moneybox ISA information.', 'Various promotional offers from Uber Eats, Schuh, and The Body Shop.']


In [96]:
print(output)

[{'type': 'time', 'hour': '05', 'minute': '36'}, {'type': 'email_summary', 'time_frame': 'Last 2 Weeks', 'total_emails_found': 50, 'bullet_points': ['Multiple job alerts received, including roles for SEO Assistant, Support Analyst, Junior Full Stack Engineer, and Data Product Owner.', 'Travel deals and booking confirmations from various services (Kiwi.com, Ryanair, TravelUp, Qatar Airways, Expedia).', 'Financial updates including account statements from Barclays and Moneybox ISA information.', 'Various promotional offers from Uber Eats, Schuh, and The Body Shop.']}]
